In [2]:
# 1: 구글 드라이브 마운트 및 7-Zip 설치
# 분할 압축(.zip.part0)을 안정적으로 풀기 위해 p7zip-full을 설치합니다.

from google.colab import drive
drive.mount('/content/drive')

!apt-get update -qq && apt-get install -y -qq p7zip-full

Mounted at /content/drive
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
# 2: download.tar 1차 압축 해제
# 드라이브의 download.tar를 /content/tar_extracted 폴더에 풉니다. 일부 항목 경로에 특수문자(?)가 있더라도 전체 실행이 멈추지 않도록 처리합니다.

import os
import subprocess
from pathlib import Path

TAR_PATH = "/content/drive/MyDrive/AIHub_Data/download.tar"  # 실제 드라이브 경로로 수정
TAR_OUT_DIR = "/content/tar_extracted"

os.makedirs(TAR_OUT_DIR, exist_ok=True)

# 1차 tar 해제 (--warning=no-unknown-keyword 등으로 진행, 일부 깨진 엔트리는 건너뜀)
cmd = ["tar", "-xvf", TAR_PATH, "-C", TAR_OUT_DIR]
proc = subprocess.run(cmd, text=True, capture_output=True)

print("tar 종료 코드:", proc.returncode)
if proc.stderr:
    print("일부 경고/오류 요약 (상위 10줄):")
    for line in proc.stderr.strip().splitlines()[:10]:
        print("  ", line)

tar 종료 코드: 0


In [ ]:
# 3: 분할 압축(.zip.part0) 2차 일괄 해제 (전체 또는 학년 필터링)
# 1차 해제 후 생성된 .zip.part0 파일들을 찾아 7-Zip으로 한 번에 풉니다.

# 전체 해제: TARGET_FILTER = ""

# 특정 학년만 해제(예: 고등 1학년): TARGET_FILTER = "08.고등학교_1학년" 또는 "고등"

import shutil

UNZIPPED_DIR = "/content/unzipped_data"
TARGET_FILTER = ""  # 예: "08.고등학교_1학년" 으로 설정하면 해당 학년만 압축 해제

os.makedirs(UNZIPPED_DIR, exist_ok=True)

part0_files = sorted(Path(TAR_OUT_DIR).rglob("*.zip.part0"))

if TARGET_FILTER:
    part0_files = [f for f in part0_files if TARGET_FILTER in str(f)]

print(f"총 {len(part0_files)}개의 .zip.part0 분할 압축 세트를 찾았습니다.")

for idx, part0 in enumerate(part0_files, 1):
    # 원본 Training / Validation 및 01.원천데이터 / 02.라벨링데이터 구조 유지
    rel_parent = part0.parent.relative_to(TAR_OUT_DIR)
    archive_folder_name = part0.name.replace(".zip.part0", "")
    dest_dir = Path(UNZIPPED_DIR) / rel_parent / archive_folder_name
    dest_dir.mkdir(parents=True, exist_ok=True)

    print(f"[{idx}/{len(part0_files)}] 압축 해제 중: {part0.name}")
    result = subprocess.run(
        ["7z", "x", str(part0), f"-o{dest_dir}", "-y"],
        text=True,
        capture_output=True
    )
    if result.returncode != 0:
        print(f"  ⚠️ 경고/오류 발생 ({part0.name}): {result.stderr.strip().splitlines()[-1] if result.stderr else ''}")

print("\n✅ 모든 압축 해제 작업이 완료되었습니다.")

SyntaxError: invalid syntax (4046559866.py, line 4)

In [ ]:
# 4: 해제 결과 요약 확인 (png / json 개수 점검)
# 원천데이터(.png)와 라벨링데이터(.json)가 제대로 풀렸는지 확인합니다.

png_files = list(Path(UNZIPPED_DIR).rglob("*.png"))
json_files = list(Path(UNZIPPED_DIR).rglob("*.json"))

print(f"추출된 PNG 이미지 수: {len(png_files):,}개")
print(f"추출된 JSON 라벨 수 : {len(json_files):,}개")

for p in png_files[:5]:
    print("  예시 PNG :", p)
for j in json_files[:5]:
    print("  예시 JSON:", j)